In [ ]:
# =========================
# BlockC — EP Flux + EP flux divergence (Global Calculation)
# =========================

import os, glob, sys
import numpy as np
import xarray as xr

# ---------------- paths ----------------
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_EP_NC   = os.path.join(OUT_DIR, "EPFLUX_BWCN002_stdplev_time_lat_ALL.nc")

# aostools path
AOSTOOLS_PY = "/mnt/data/aostools_functions.py"

# ---------------- standard pressure levels (Pa) ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# EP-flux options
DO_UBAR = False   
WAVE    = -1      

# ---------------- import ComputeEPfluxDiv ----------------
if os.path.exists(AOSTOOLS_PY):
    sys.path.insert(0, os.path.dirname(AOSTOOLS_PY))
from aostools_functions import ComputeEPfluxDiv

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]  # (lev)
    P0   = ds["P0"]    # scalar
    PS   = ds["PS"]    # (time, lat, lon)
    # xarray broadcasting will handle dimensions automatically
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    return p_mid

def interp_profile_logp_4d(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: np.ndarray) -> xr.DataArray:
    """
    4D 插值: (time, lev, lat, lon) -> (time, plev, lat, lon)
    利用 xarray 的 apply_ufunc 自动广播 time, lat, lon 维度
    """
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        # 这是一个针对单列 (lev) 的函数，apply_ufunc 会自动对所有 grid 执行它
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)
        return np.interp(np.log(p_tgt_pa), np.log(p_use[idx]), v_use[idx], left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,         # 关键：这会对所有非 core 维度（time, lat, lon）进行循环
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": len(p_tgt_pa)},
    )
    out = out.assign_coords(plev=("plev", p_tgt_pa))
    return out

# ---------------- main ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
print(f"[BlockC_EP] Found {len(h3_files)} h3 files.")

# 1. Open Dataset (Lazy)
ds = xr.open_mfdataset(
    h3_files,
    combine="by_coords",
    parallel=True,
    use_cftime=True,
    chunks={"time": 12}, # 稍微大一点的 chunk 减少小文件开销
    data_vars="minimal",
    coords="minimal"
)

# 2. Resample to Monthly Mean
print("[BlockC_EP] Resampling to monthly mean...")
ds_m = ds[["U","V","T","OMEGA","PS","P0","hyam","hybm"]].resample(time="MS").mean()

# 3. Load Data into Memory (The "All at Once" step)
#    注意：这里会触发大量计算和内存占用。确保内存足够。
print("[BlockC_EP] Loading all data into memory (This may take a while)...")
ds_loaded = ds_m.load()

# 4. Compute 3D Pressure Field
print("[BlockC_EP] Computing 3D pressure field...")
p_mid = compute_pressure_mid(ds_loaded) # (time, lev, lat, lon)

# 5. Interpolate to Standard Levels (Time-Vectorized)
print("[BlockC_EP] Interpolating variables to standard pressure levels...")
# 结果形状: (time, plev, lat, lon)
u_std = interp_profile_logp_4d(ds_loaded["U"], p_mid, PLEV_STD_PA)
v_std = interp_profile_logp_4d(ds_loaded["V"], p_mid, PLEV_STD_PA)
t_std = interp_profile_logp_4d(ds_loaded["T"], p_mid, PLEV_STD_PA)
w_std = interp_profile_logp_4d(ds_loaded["OMEGA"]/100.0, p_mid, PLEV_STD_PA) # Pa/s -> hPa/s

# 准备 numpy 数组供 aostools 使用
# transpose 确保顺序为 (time, plev, lat, lon)
# .values 将其转为纯 numpy 数组
u_np = u_std.transpose("time", "plev", "lat", "lon").values
v_np = v_std.transpose("time", "plev", "lat", "lon").values
t_np = t_std.transpose("time", "plev", "lat", "lon").values
w_np = w_std.transpose("time", "plev", "lat", "lon").values
lat_np = ds_loaded["lat"].values
plev_hpa = (PLEV_STD_PA / 100.0)

print(f"[BlockC_EP] Starting EP Flux calculation for shape: {u_np.shape}...")

# 6. Compute EP Flux (Unified)
#    输入已经是 (time, p, lat, lon)，不需要额外增加维度
ep1_cart, ep2_cart, div1, div2 = ComputeEPfluxDiv(
    lat=lat_np,
    pres=plev_hpa,
    u=u_np,
    v=v_np,
    t=t_np,
    w=w_np,
    do_ubar=DO_UBAR,
    wave=WAVE,
)

# 结果形状应为 (time, plev, lat)
div_total = div1 + div2
print(f"[BlockC_EP] Calculation done. Output shape: {div_total.shape}")

# 7. Save to NetCDF
out = xr.Dataset(
    data_vars=dict(
        EP1_cart   = (("time","plev","lat"), ep1_cart,
                      {"units":"m2 s-2",      "long_name":"Meridional EP flux (cartesian)"}),
        EP2_cart   = (("time","plev","lat"), ep2_cart,
                      {"units":"hPa m s-2",  "long_name":"Vertical EP flux (cartesian)"}),
        DIV1       = (("time","plev","lat"), div1,
                      {"units":"m s-1 day-1","long_name":"Horizontal EP-flux divergence"}),
        DIV2       = (("time","plev","lat"), div2,
                      {"units":"m s-1 day-1","long_name":"Vertical EP-flux divergence"}),
        DIV_TOTAL  = (("time","plev","lat"), div_total,
                      {"units":"m s-1 day-1","long_name":"Total EP-flux divergence"}),
    ),
    coords=dict(
        time = ds_loaded["time"],
        plev = ("plev", PLEV_STD_PA, {"units":"Pa"}),
        lat  = ds_loaded["lat"],
    ),
    attrs=dict(source="Global calculation via aostools")
)

encoding = {vn: {"zlib": True, "complevel": 4, "_FillValue": np.nan} for vn in out.data_vars}
out.to_netcdf(OUT_EP_NC, encoding=encoding)
print("[BlockC_EP] Saved:", OUT_EP_NC)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

# ================= 配置区域 =================
# 输入文件路径 (BlockC 的输出)
INPUT_FILE = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_time_lat_ALL.nc"

# 绘图参数
LAT_RANGE = (40, 80)        # 纬度范围
LEV_RANGE = (1000, 1)        # 气压范围 (hPa)
TIME_RANGE = ("0007-06-01", "0008-10-31") # 时间范围

# ================= 1. 读取数据 =================
print(f"读取文件: {INPUT_FILE}")
ds = xr.open_dataset(INPUT_FILE, use_cftime=True)

# 将气压层从 Pa 转换为 hPa，方便处理
ds["plev"] = ds["plev"] / 100.0
ds["plev"].attrs["units"] = "hPa"

# ================= 2. 纬度加权平均 (40N - 80N) =================
# 截取纬度
# 注意：slice 的顺序取决于 lat 是递增还是递减，这里使用 min/max 确保安全
ds_sub = ds.sel(lat=slice(min(LAT_RANGE), max(LAT_RANGE)))

# 计算纬度权重 (cos(lat))
weights = np.cos(np.deg2rad(ds_sub.lat))
weights.name = "weights"

# 对 EP2_cart (Fz) 进行加权平均
# 结果维度: (time, plev)
fz_mean = ds_sub["EP2_cart"].weighted(weights).mean(dim="lat")

# ================= 3. 计算标准化异常 (Standardized Anomaly) =================
print("计算标准化异常 (Standardized Anomaly)...")

# 1. 计算气候态平均值 (Mean)
clim_mean = fz_mean.groupby("time.month").mean("time")

# 2. 计算气候态标准差 (Std Dev)
clim_std = fz_mean.groupby("time.month").std("time")

# 3. 计算标准化异常
# 先减去均值
anom_raw = fz_mean.groupby("time.month") - clim_mean

# 再除以标准差 (注意：需要再次 groupby 以确保月份对齐)
fz_std_anom = anom_raw.groupby("time.month") / clim_std

# 更新绘图数据变量名
plot_data = fz_std_anom.sel(plev=slice(min(LEV_RANGE), max(LEV_RANGE)))
plot_data = plot_data.sel(time=slice(*TIME_RANGE))

# ================= 5. 绘图 (修正版) =================
fig, ax = plt.subplots(figsize=(10, 6))

# 设置 Time - Log(Pressure) 坐标
divnorm = mcolors.TwoSlopeNorm(vmin=plot_data.min(), vcenter=0, vmax=plot_data.max())

# --- 关键修改开始 ---
# 1. 创建一个临时的数字 X 轴 (0, 1, 2, ..., N-1)
x_numeric = np.arange(len(plot_data.time))

# 建议手动指定 levels，因为标准化异常通常在这个范围
levels_std = np.linspace(-3.0, 3.0, 13) # 从 -3 到 3，每隔 0.5 一个级

cf = ax.contourf(
    x_numeric, 
    plot_data.plev, 
    plot_data.transpose("plev", "time"),
    levels=levels_std, # <--- 使用固定的 sigma 级别
    cmap="RdBu_r",
    extend="both"
)

# 3. 手动设置 X 轴刻度 (让它看起来像时间)
# 我们每隔几个点取一个刻度，避免太挤。例如每隔 3 个月显示一次。
step = 3  
if len(x_numeric) > 24: step = 6 # 如果时间很长，间隔大一点

# 选出要显示刻度的位置 (数字索引)
tick_indices = x_numeric[::step]

# 将这些位置对应的 cftime 对象转为字符串 (例如 "Year 7-06")
# 这里的格式可以自己改，比如 t.strftime("%Y-%m")
tick_labels = [t.strftime("%Y-%m") for t in plot_data.time.values[::step]]

ax.set_xticks(tick_indices)
ax.set_xticklabels(tick_labels, rotation=45) # 旋转45度防止重叠
# --- 关键修改结束 ---

# 设置 Y 轴为对数坐标并反转
ax.set_yscale("log")
ax.invert_yaxis()

# 设置 Y 轴刻度格式
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_yticks([1, 2, 5, 10, 20, 50, 100, 300, 1000])
ax.set_yticklabels(["1", "2", "5", "10", "20", "50", "100", "300", "1000"])

# 装饰图片
plt.colorbar(cf, ax=ax, label="Vertical EP Flux Anomaly (hPa m s$^{-2}$)")
ax.set_title(f"Vertical EP Flux Anomalies ({LAT_RANGE[0]}-{LAT_RANGE[1]}°N)")
ax.set_ylabel("Pressure (hPa)")
ax.set_xlabel("Model Time")

plt.tight_layout()
plt.show()